# 04 · Data cleaning: organizations (`Organization.xlsx`)

**Goal:** one clean reference table of every partner organization: food shelves, farmers markets, meal programs, agencies and food banks. It gives each one a **type**, **county + FIPS**, **food bank**, **status** and **map location**, so other tables (food shelf stats, Market Bucks, calls) can be joined to geography.

**Input:** `Data/Organization.xlsx` (1,202 rows × 68 columns, one row per organization record) and `Data/processed/mmg_mn_county_2022_2024.csv` (official county list + FIPS).

**Output (`Data/processed/`):** `organization_clean.csv`, one row per organization.

**Join key:** `org_id` = the raw `__ID`. Keep it as a number **with decimals**: some IDs have decimals (e.g. `14681` and `14681.1` are **different** sites). Never round it. It matches `site_id` in `foodshelf_site_month.csv` and `_id_site` in Market Bucks.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "Data"
OUT_DIR = DATA / "processed"
mmg = pd.read_csv(OUT_DIR / "mmg_mn_county_2022_2024.csv", dtype={"fips": str})
COUNTY_FIPS = dict(zip(mmg["county"], mmg["fips"]))      # official 87 counties -> FIPS
FIPS_COUNTY = {v: k for k, v in COUNTY_FIPS.items()}

raw = pd.read_excel(DATA / "Organization.xlsx")
print("raw:", raw.shape, "| __ID unique:", raw["__ID"].is_unique, "| IDs with decimals:", int((raw["__ID"] % 1 != 0).sum()))

raw: (1202, 68) | __ID unique: True | IDs with decimals: 13


## Step 1: Keep useful columns

- **What:** keep IDs, type, location, county, status, food bank, TEFAP, SuperShelf and FindHelp fields, renamed to readable names.
- **Dropped and why:**
  - system/audit timestamps (`zz…`, `CreateTime`);
  - mailing address (the physical address is what matters);
  - legislative districts and tax-ID verification;
  - legacy flags `MFS` / `MCS` (1% filled);
  - `Eligibility` / `Conditions` (free text, only ~2% filled);
  - `zcountDaysOperations` (almost always 7, so it carries no information);
  - the `z12MoAverage_…` columns (a rolling snapshot as of the extract date; real monthly activity is in `foodshelf_site_month.csv`).

In [2]:
COLS = {
    "__ID": "org_id", "AgencyType": "agency_type", "AgencySubType": "agency_subtype",
    "City": "city", "State": "state", "Zip": "zip", "County": "county_original",
    "Region": "region", "SuperRegion": "super_region",
    "ActiveInactive": "status", "InactiveDate": "inactive_date",
    "_id_Bank": "food_bank_id", "_id_Agency": "parent_agency_id", "isDistributionSite": "is_distribution_site",
    "TefapStatus": "tefap_status", "SuperShelfStatus": "supershelf_status", "SuperShelfCertDate": "supershelf_cert_date",
    "FindHelpMapCategory": "findhelp_category", "MarketSeasons": "market_seasons",
    "block_geoid": "block_geoid", "is_zip_centroid": "is_zip_centroid",
}
org = raw[list(COLS)].rename(columns=COLS)
org["agency_type"] = org["agency_type"].str.strip()
print(org.shape)
org.head(3)

(1202, 21)


,org_id,agency_type,agency_subtype,city,state,zip,county_original,region,super_region,status,...,food_bank_id,parent_agency_id,is_distribution_site,tefap_status,supershelf_status,supershelf_cert_date,findhelp_category,market_seasons,block_geoid,is_zip_centroid
0,1.0,Agency or Tribal Nation,NaN,Rochester,MN,55904.0,Olmsted,Southern Minnesota,Non-Metro,Active,...,1.0,1.0,0.0,Active,NaN,NaT,NaN,NaN,2.710900e+14,0.0
1,3.0,Food Bank,NaN,Brooklyn Park,MN,55428.0,Hennepin-Suburbs,Twin Cities 7-Metro,Metro,Active,...,3.0,15750.0,0.0,NaN,NaN,NaT,Food Shelves,NaN,2.705303e+14,0.0
2,5.0,Food Bank,NaN,East Grand Forks,MN,56721.0,Polk,Northern Minnesota,Non-Metro,Active,...,5.0,5.0,0.0,Active,NaN,NaT,NaN,NaN,2.711902e+14,0.0


## Step 2: Organization type → `org_group`

- **What:** map the 9 `AgencyType` labels into groups. Food shelf labels use the **same grouping as `foodshelf_site_month.csv`** (`site_group`), so the two files agree.
- **Why:** questions need "food shelves only" or "farmers markets only" without retyping label lists.
- **Note:** `Agency or Tribal Nation` rows are mostly **parent organizations** that run one or more sites (`parent_agency_id` on the site rows points to them), not distribution points themselves.

In [3]:
ORG_GROUP = {
    "Brick and Mortar Site": "food_shelf", "College Campus Program": "food_shelf", "Tribal Program": "food_shelf",
    "Outreach Program": "outreach", "Meal Program": "meal", "Farmers Market": "farmers_market",
    "Food Bank": "food_bank", "Agency or Tribal Nation": "agency", "Other": "other",
}
org["org_group"] = org["agency_type"].map(ORG_GROUP).fillna("unknown")
assert org.loc[org["agency_type"].notna(), "org_group"].ne("unknown").all(), "an AgencyType label is not mapped"
org.groupby(["org_group", "agency_type"], dropna=False).size().rename("organizations").to_frame()

organizations
org_group      agency_type                           
agency         Agency or Tribal Nation            317
farmers_market Farmers Market                     185
food_bank      Food Bank                            5
food_shelf     Brick and Mortar Site              448
               College Campus Program               6
               Tribal Program                       6
meal           Meal Program                        71
other          Other                               38
outreach       Outreach Program                    93
unknown        NaN                                 33

## Step 3: Fix state and ZIP

- **What:**
  - Trim whitespace in `state` (`"MN\n"` → `MN`).
  - Three rows have a **ZIP code typed into the state box** (e.g. `55313`). Move it to `zip` when `zip` is empty, and set the state to `MN`.
  - Store ZIP as **5-digit text**; ZIP+4 values such as `563420344` keep their first 5 digits.
- **Why:** ZIPs are labels, not numbers. As numbers they lose leading zeros and can't be joined to ZIP-level public data.

In [4]:
state_raw = org["state"].astype("string").str.strip()
zip_in_state = state_raw.str.fullmatch(r"5[5-6]\d{3}").fillna(False)
print("ZIP typed into the state column:", int(zip_in_state.sum()), "| of those with an empty zip:",
      int((zip_in_state & org["zip"].isna()).sum()))
org.loc[zip_in_state & org["zip"].isna(), "zip"] = state_raw[zip_in_state & org["zip"].isna()].astype(float)
org["state"] = state_raw.mask(zip_in_state, "MN")

zip_num = org["zip"].astype(float)
zip_plus4 = zip_num > 99999
print("ZIP+4 values shortened to 5 digits:", int(zip_plus4.sum()))
zip_num = zip_num.where(~zip_plus4, (zip_num // 10000))
org["zip"] = zip_num.map(lambda v: f"{int(v):05d}" if pd.notna(v) else None)
print("state values:", org["state"].value_counts(dropna=False).to_dict())
print("non-Minnesota ZIPs (not 55xxx/56xxx):", org.loc[org["zip"].notna() & ~org["zip"].fillna("").str.match(r"5[5-6]"), ["org_id", "city", "state", "zip"]].values.tolist())

ZIP typed into the state column: 3 | of those with an empty zip: 3
ZIP+4 values shortened to 5 digits: 9
state values: {'MN': 1061, <NA>: 140, 'ND': 1}
non-Minnesota ZIPs (not 55xxx/56xxx): [[7.0, 'Fargo', 'ND', '58102']]


## Step 4: County → 87 official counties + FIPS

- **What:**
  1. Merge city/suburb splits and fix spellings, the **same rules as the food shelf cleaning**: `Hennepin-Minneapolis` / `Hennepin-Suburbs` → Hennepin, `Ramsey-Saint Paul` / `Ramsey-Suburbs` → Ramsey, `Saint Louis` → St. Louis, `Lac Qui Parle` → Lac qui Parle, `Mcleod` → McLeod.
  2. Where `County` is blank but the record has a Census block, take the county from the block (its first 5 digits are the county FIPS). Recorded in `county_source`.
  3. Flag records whose block county differs from the stated county (`flag_block_county_differs`). Usually a border town or an office address in another county; both values are kept.
- **Why:** FIPS is how every other dataset joins; the city/suburb split exists in no public dataset.

In [5]:
def standardize(name: pd.Series) -> pd.Series:
    s = name.str.strip().str.replace(r"-(Minneapolis|Suburbs|Saint Paul)$", "", regex=True)
    return s.replace({"Saint Louis": "St. Louis", "Lac Qui Parle": "Lac qui Parle", "Mcleod": "McLeod"})

org["block_geoid"] = org["block_geoid"].map(lambda v: f"{int(v):015d}" if pd.notna(v) else None)
org["block_fips"] = org["block_geoid"].str[:5]

org["county"] = standardize(org["county_original"])
org["fips"] = org["county"].map(COUNTY_FIPS)
unmatched = org.loc[org["county"].notna() & org["fips"].isna(), "county"].unique()
assert len(unmatched) == 0, f"county names with no FIPS: {unmatched}"
org["county_source"] = np.where(org["fips"].notna(), "Organization", None)

fill = org["fips"].isna() & org["block_fips"].isin(FIPS_COUNTY)
org.loc[fill, "fips"] = org.loc[fill, "block_fips"]
org.loc[fill, "county"] = org.loc[fill, "fips"].map(FIPS_COUNTY)
org.loc[fill, "county_source"] = "census block"

org["flag_block_county_differs"] = org["block_fips"].notna() & org["fips"].notna() & (org["block_fips"] != org["fips"])
print(f"county blank in raw: {org['county_original'].isna().sum()} | filled from census block: {int(fill.sum())} "
      f"| still unknown: {org['fips'].isna().sum()}")
print("still-unknown county by type:", org.loc[org["fips"].isna(), "org_group"].value_counts().to_dict())
print("records whose block is in a different county:", int(org["flag_block_county_differs"].sum()))
print("counties covered:", org["fips"].nunique(), "of 87")

county blank in raw: 62 | filled from census block: 10 | still unknown: 52
still-unknown county by type: {'farmers_market': 34, 'unknown': 11, 'agency': 5, 'food_shelf': 1, 'outreach': 1}
records whose block is in a different county: 37
counties covered: 86 of 87


## Step 5: Food bank name, status flags

- **What:**
  - `food_bank_id` points to the food bank's own organization record. Turn it into a readable `food_bank` name, with a stray `*` and the suffix `Inc.` removed. The food shelf data itself switches between "North Country Food Bank" and "North Country Food Bank Inc." over time, so one clean spelling is used. **When joining to `foodshelf_site_month.csv`, remove `*` and ` Inc.` from its `food_bank` too.**
  - ID `7` is an **inactive** food bank record in Fargo, ND with no name. Its name is taken from the food shelf data, which calls those sites' food bank "Great Plains Food Bank" (evidence printed below). ID `-1` is not a food bank, so it becomes blank.
  - `is_active` = `status == "Active"`.
  - SuperShelf: one stray value `1` becomes blank; only `Certified` / `In Process` are valid.
- **Why:** questions compare food bank networks and service models; status matters when counting current sites.

In [6]:
def clean_bank(name: pd.Series) -> pd.Series:
    return name.str.replace("*", "", regex=False).str.replace(r"\s+Inc\.?$", "", regex=True).str.strip()

bank_names = clean_bank(raw.loc[raw["zNameIfActiveBank"].notna()].set_index("__ID")["zNameIfActiveBank"])
# Food bank IDs without a name (inactive records): use the name the food shelf data gives those organizations' sites
site_fb = (pd.read_csv(OUT_DIR / "foodshelf_site_month.csv", usecols=["site_id", "month", "food_bank"])
             .sort_values("month").drop_duplicates("site_id", keep="last")          # each site's latest food bank
             .assign(food_bank=lambda d: clean_bank(d["food_bank"])))
for bank_id in sorted(set(org["food_bank_id"].dropna()) - set(bank_names.index) - {-1}):
    names = site_fb.loc[site_fb["site_id"].isin(org.loc[org["food_bank_id"] == bank_id, "org_id"]), "food_bank"].value_counts()
    print(f"food_bank_id {bank_id:g}: names its sites have in the food shelf data -> {names.to_dict()}")
    if len(names) == 1:
        bank_names.loc[bank_id] = names.index[0]
print("food banks:", bank_names.to_dict())
org["food_bank"] = org["food_bank_id"].map(bank_names)
check = org[["org_id", "food_bank"]].merge(site_fb, left_on="org_id", right_on="site_id", suffixes=("", "_site"))
print(f"agreement with the food shelf data's food bank name: {(check['food_bank'] == check['food_bank_site']).mean():.1%} "
      f"of {len(check)} sites")

org["is_active"] = org["status"].eq("Active")
org["supershelf_status"] = org["supershelf_status"].where(org["supershelf_status"].isin(["Certified", "In Process"]))
org["is_zip_centroid"] = org["is_zip_centroid"].map({1: True, 0: False})
org["is_distribution_site"] = org["is_distribution_site"].map({1: True, 0: False})
org.groupby("org_group")["food_bank"].value_counts().unstack(fill_value=0)

food_bank_id 7: names its sites have in the food shelf data -> {'Great Plains Food Bank': 7}
food banks: {1.0: 'Channel One Regional Food Bank', 3.0: 'Second Harvest Heartland', 5.0: 'North Country Food Bank', 6.0: 'Second Harvest Northland', 14179.0: 'The Food Group', 7.0: 'Great Plains Food Bank'}
agreement with the food shelf data's food bank name: 100.0% of 624 sites


food_bank,Channel One Regional Food Bank,Great Plains Food Bank,North Country Food Bank,Second Harvest Heartland,Second Harvest Northland,The Food Group
org_group,,,,,,
agency,44,0,54,126,33,54
food_bank,0,1,1,1,1,0
food_shelf,59,2,61,170,66,102
meal,0,0,17,17,12,19
outreach,2,5,0,48,35,1
unknown,0,0,2,9,4,0


## Step 6: Map location (latitude / longitude)

- **What:** look up the **center point of each organization's 2020 Census block** with the public Census TIGERweb service. Results are cached in `Data/local/` (git-ignored), so this only needs the internet once.
- **Why:** distance to the nearest food shelf or farmers market is a key access measure (Q1, Q6). The block is the finest location the data gives; the street address was replaced by the block before release.
- **Caveat:** 12 records were located only by their ZIP centre (`is_zip_centroid = True`), so their point is approximate.

In [7]:
import requests

LOCAL = DATA / "local"
LOCAL.mkdir(exist_ok=True)
coords_file = LOCAL / "organization_block_coords.csv"
TIGER_URL = "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/tigerWMS_Census2020/MapServer/10/query"

blocks = sorted(org["block_geoid"].dropna().unique())
cached = pd.read_csv(coords_file, dtype={"GEOID": str}) if coords_file.exists() else pd.DataFrame(columns=["GEOID", "lat", "lon"])
todo = [b for b in blocks if b not in set(cached["GEOID"])]
fetched = []
for i in range(0, len(todo), 50):
    query = "GEOID IN (%s)" % ",".join(f"'{b}'" for b in todo[i:i + 50])
    reply = requests.get(TIGER_URL, params={"where": query, "outFields": "GEOID,INTPTLAT,INTPTLON",
                                            "returnGeometry": "false", "f": "json"}, timeout=60).json()
    fetched += [{"GEOID": f["attributes"]["GEOID"], "lat": float(f["attributes"]["INTPTLAT"]),
                 "lon": float(f["attributes"]["INTPTLON"])} for f in reply["features"]]
new_xy = pd.DataFrame(fetched, columns=["GEOID", "lat", "lon"])
block_xy = pd.concat([f for f in (cached, new_xy) if len(f)], ignore_index=True).drop_duplicates("GEOID")
block_xy.to_csv(coords_file, index=False)

org = org.merge(block_xy.rename(columns={"GEOID": "block_geoid"}), on="block_geoid", how="left", validate="many_to_one")
print(f"blocks: {len(blocks)} | located: {org['lat'].notna().sum()} of {len(org)} records "
      f"| block not found: {len(set(blocks) - set(block_xy['GEOID']))}")

blocks: 690 | located: 1052 of 1202 records | block not found: 0


## Step 7: Link to food shelf activity, check and save

- `has_foodshelf_stats` = the organization appears in `foodshelf_site_month.csv` (it reported monthly food shelf numbers at some point).

In [8]:
site_ids = set(pd.read_csv(OUT_DIR / "foodshelf_site_month.csv", usecols=["site_id"])["site_id"])
org["has_foodshelf_stats"] = org["org_id"].isin(site_ids)
print("organizations with food shelf stats:", int(org["has_foodshelf_stats"].sum()),
      "| food shelf-stat sites missing from Organization:", len(site_ids - set(org["org_id"])))

ORDER = ["org_id", "org_group", "agency_type", "agency_subtype", "status", "is_active", "inactive_date",
         "county", "fips", "county_source", "county_original", "city", "state", "zip", "region", "super_region",
         "food_bank", "food_bank_id", "parent_agency_id", "is_distribution_site", "has_foodshelf_stats",
         "tefap_status", "supershelf_status", "supershelf_cert_date", "findhelp_category", "market_seasons",
         "block_geoid", "is_zip_centroid", "lat", "lon", "flag_block_county_differs"]
org = org[ORDER].sort_values("org_id").reset_index(drop=True)
assert org["org_id"].is_unique and len(org) == len(raw)
org.to_csv(OUT_DIR / "organization_clean.csv", index=False)
print("saved organization_clean.csv:", org.shape)
org.groupby("org_group").agg(organizations=("org_id", "size"), active=("is_active", "sum"),
                             with_county=("fips", lambda s: s.notna().sum()), with_location=("lat", lambda s: s.notna().sum()))

organizations with food shelf stats: 624 | food shelf-stat sites missing from Organization: 0
saved organization_clean.csv: (1202, 31)


,organizations,active,with_county,with_location
org_group,,,,
agency,317,307,312,287
farmers_market,185,179,151,97
food_bank,5,4,5,5
food_shelf,460,410,459,445
meal,71,49,71,71
other,38,38,38,38
outreach,93,63,92,87
unknown,33,23,22,22


## Summary

- **1,202 organizations, 31 columns** saved to `organization_clean.csv`.
- **Types:** 460 food shelf sites, 317 agencies (mostly parent organizations), 185 farmers markets, 93 outreach programs, 71 meal programs, 5 food bank records, 38 other, 33 untyped.
- **Fixes:** 3 ZIPs moved out of the state column, 9 ZIP+4 values shortened, county names merged to the 87 official counties (10 filled from the Census block).
- **County unknown for 52 records**, mostly farmers markets (34) with no address at all.
- **Food bank names:** "Great Plains Food Bank" assigned to ID 7; names agree 100% with `foodshelf_site_month.csv` (after removing `*` / `Inc.`).
- **Map location** (block center) for 1,052 records; 37 records have a block in a different county than stated (flagged, both kept).
- **All 624 sites** in `foodshelf_site_month.csv` are in this table (`has_foodshelf_stats`).